# Project 3 - Credit Risk Modeling (Balanced v2)

Notebook renforcé avec tes remarques:
- analyse des données plus claire (EDA)
- visualisation explicite de la cible
- section Lasso + Random Forest pour analyser l'importance des variables
- gestion du déséquilibre explicitée (class weights + SMOTE + scale_pos_weight)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Optional dependencies
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except Exception as exc:
    raise RuntimeError("Install imbalanced-learn: pip install imbalanced-learn") from exc

try:
    from xgboost import XGBClassifier
except Exception as exc:
    raise RuntimeError("Install xgboost: pip install xgboost") from exc

In [ ]:
# Configuration
OUTPUT_DIR = Path('outputs')
CM_DIR = OUTPUT_DIR / 'confusion_matrices'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
DEFAULT_THRESHOLD = 0.50
CV_FOLDS = 2  # set 3 if runtime is acceptable

# Data loading: Colab upload or local fallback
try:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No uploaded file. Please run files.upload() again.")
    uploaded_filename = next(iter(uploaded.keys()))
    DATA_PATH = Path(uploaded_filename)
    print(f"Data loaded via Colab upload: {DATA_PATH}")
except ImportError:
    DATA_PATH = Path('data/default_of_credit_card_clients.xls')
    print(f"google.colab not found, local fallback path: {DATA_PATH}")

In [ ]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(' ', '_')
        .replace('.', '_')
        .replace('/', '_')
        .replace('-', '_')
    )


def load_credit_dataset(data_path: Path) -> pd.DataFrame:
    if not data_path.exists():
        raise FileNotFoundError(f'Dataset not found: {data_path}')

    if data_path.suffix.lower() == '.csv':
        df = pd.read_csv(data_path)
    elif data_path.suffix.lower() in {'.xls', '.xlsx'}:
        try:
            df = pd.read_excel(data_path)
            if 'default.payment.next.month' not in [str(c) for c in df.columns]:
                df = pd.read_excel(data_path, header=1)
        except Exception:
            df = pd.read_excel(data_path, header=1)
    else:
        raise ValueError('Unsupported format. Use .csv/.xls/.xlsx')

    df = df.loc[:, ~df.columns.astype(str).str.contains('^Unnamed')]
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


def detect_target_column(columns: list[str]) -> str:
    for c in ['default_payment_next_month', 'default', 'y', 'target']:
        if c in columns:
            return c
    raise ValueError('Target column not found (expected default.payment.next.month)')


def add_feature_engineering(df: pd.DataFrame):
    df = df.copy()
    created = []

    bill_cols = [f'bill_amt{i}' for i in range(1, 7) if f'bill_amt{i}' in df.columns]
    pay_amt_cols = [f'pay_amt{i}' for i in range(1, 7) if f'pay_amt{i}' in df.columns]
    pay_status_cols = [c for c in df.columns if c.startswith('pay_') and not c.startswith('pay_amt')]

    if len(bill_cols) > 0:
        df['bill_amt_mean'] = df[bill_cols].mean(axis=1)
        created.append('bill_amt_mean')

    if len(pay_amt_cols) > 0:
        df['pay_amt_mean'] = df[pay_amt_cols].mean(axis=1)
        created.append('pay_amt_mean')

    if len(bill_cols) > 0 and len(pay_amt_cols) > 0:
        df['pay_to_bill_total_ratio'] = (
            df[pay_amt_cols].sum(axis=1) / (df[bill_cols].abs().sum(axis=1) + 1)
        )
        created.append('pay_to_bill_total_ratio')

    if len(bill_cols) > 0 and 'limit_bal' in df.columns:
        util_matrix = df[bill_cols].div(df['limit_bal'].replace(0, np.nan), axis=0)
        util_matrix = util_matrix.replace([np.inf, -np.inf], np.nan)
        df['util_ratio_mean'] = util_matrix.mean(axis=1).fillna(0)
        df['util_ratio_max'] = util_matrix.max(axis=1).fillna(0)
        created += ['util_ratio_mean', 'util_ratio_max']

    if len(pay_status_cols) > 0:
        df['delinquency_count'] = (df[pay_status_cols] > 0).sum(axis=1)
        df['delinquency_max'] = df[pay_status_cols].max(axis=1)
        created += ['delinquency_count', 'delinquency_max']

    if len(bill_cols) >= 2:
        df['bill_trend_1_vs_6'] = df[bill_cols[0]] - df[bill_cols[-1]]
        created.append('bill_trend_1_vs_6')

    return df, sorted(set(created))

## 1) Data Loading + Initial Checks

In [ ]:
df = load_credit_dataset(DATA_PATH)
target_col = detect_target_column(list(df.columns))

# Drop ID-like columns if present
id_cols = [c for c in df.columns if c in {'id', 'id_'} or c.startswith('id_')]
if id_cols:
    df = df.drop(columns=id_cols)

df, created_features = add_feature_engineering(df)

print('Shape:', df.shape)
print('Target:', target_col)
print('Created engineered features:', created_features)

display(df.head())

## 2) Data Exploration (EDA)

In [ ]:
# Data quality
print('Data types:')
display(df.dtypes.value_counts().rename('count').to_frame())

print('Duplicate rows:', int(df.duplicated().sum()))

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print('Missing values:')
if len(missing) == 0:
    print('No missing values detected.')
else:
    missing_df = pd.DataFrame({
        'missing_count': missing,
        'missing_ratio_pct': (100 * missing / len(df)).round(2),
    })
    display(missing_df)

print('Descriptive statistics (key variables):')
key_stats = [
    c for c in [
        'limit_bal', 'age', 'bill_amt_mean', 'pay_amt_mean',
        'util_ratio_mean', 'pay_to_bill_total_ratio', 'delinquency_count'
    ] if c in df.columns
]
if key_stats:
    display(df[key_stats].describe().T)

## 3) Visualisation de la Cible

In [ ]:
# Target visualization
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.countplot(x=df[target_col], ax=axes[0])
axes[0].set_title('Target count (0=non-default, 1=default)')
axes[0].set_xlabel('Target')

counts = df[target_col].value_counts().sort_index()
axes[1].pie(counts.values, labels=[f'{i}' for i in counts.index], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Target ratio')

plt.tight_layout()
plt.show()

print('Target distribution table:')
target_dist = pd.DataFrame({
    'count': df[target_col].value_counts().sort_index(),
    'ratio': df[target_col].value_counts(normalize=True).sort_index().round(4),
})
display(target_dist)

default_rate = float(df[target_col].mean())
print(f'Default rate = {default_rate:.2%}')

## 4) EDA Ciblée: Défaut vs Non-Défaut

In [ ]:
compare_cols = [
    c for c in [
        'limit_bal', 'age', 'bill_amt_mean', 'pay_amt_mean',
        'util_ratio_mean', 'pay_to_bill_total_ratio', 'delinquency_count', 'delinquency_max'
    ] if c in df.columns
]

rows = []
for c in compare_cols:
    mean_0 = df.loc[df[target_col] == 0, c].mean()
    mean_1 = df.loc[df[target_col] == 1, c].mean()
    pct_diff = ((mean_1 - mean_0) / (abs(mean_0) + 1e-9)) * 100
    rows.append({
        'feature': c,
        'mean_non_default': mean_0,
        'mean_default': mean_1,
        'pct_diff_default_vs_non_default': pct_diff,
    })

comparison_df = pd.DataFrame(rows).sort_values(
    'pct_diff_default_vs_non_default',
    key=lambda s: s.abs(),
    ascending=False,
)

display(comparison_df)
comparison_df.to_csv(OUTPUT_DIR / 'eda_default_vs_non_default.csv', index=False)
print('Saved:', OUTPUT_DIR / 'eda_default_vs_non_default.csv')

In [ ]:
# Correlation with target + compact heatmap
numeric_df = df.select_dtypes(include=[np.number]).copy()
if target_col in numeric_df.columns:
    corr_target = (
        numeric_df.corr(numeric_only=True)[target_col]
        .drop(target_col)
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )

    print('Top correlations with target:')
    display(corr_target.head(12).to_frame('corr_with_target'))

    top_cols = corr_target.head(8).index.tolist() + [target_col]
    plt.figure(figsize=(8, 5))
    sns.heatmap(numeric_df[top_cols].corr(numeric_only=True), cmap='coolwarm', center=0)
    plt.title('Correlation heatmap (top features + target)')
    plt.tight_layout()
    plt.show()

## 5) Split + Gestion du Déséquilibre

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

# Class weights computed explicitly
classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print('Computed class weights:', class_weight_dict)

# XGBoost imbalance parameter
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / max(pos, 1)
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

# SMOTE for MLP (and optional experiments)
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print('After SMOTE:', pd.Series(y_train_smote).value_counts().to_dict())

## 6) Lasso + RandomForest (Feature Relevance)

In [ ]:
# Lasso-style selection via L1 Logistic Regression
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        C=0.1,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        max_iter=2000,
    )),
])
lasso_pipe.fit(X_train, y_train)
lasso_coef = pd.Series(
    lasso_pipe.named_steps['model'].coef_[0],
    index=X_train.columns,
)

non_zero = (lasso_coef != 0).sum()
print(f'L1 Logistic kept {non_zero}/{len(lasso_coef)} features (non-zero coefficients).')

top_lasso = lasso_coef.abs().sort_values(ascending=False).head(12).to_frame('abs_coef')
print('Top L1 features:')
display(top_lasso)

# Random Forest feature importance (quick model)
rf_selector = RandomForestClassifier(
    n_estimators=180,
    max_depth=8,
    min_samples_leaf=5,
    class_weight=class_weight_dict,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_selector.fit(X_train, y_train)
rf_imp = pd.Series(rf_selector.feature_importances_, index=X_train.columns)

top_rf = rf_imp.sort_values(ascending=False).head(12).to_frame('importance')
print('Top Random Forest features:')
display(top_rf)

feature_screen_df = pd.DataFrame({
    'feature': X_train.columns,
    'lasso_abs_coef': lasso_coef.abs().values,
    'rf_importance': rf_imp.values,
}).sort_values(['rf_importance', 'lasso_abs_coef'], ascending=False)

feature_screen_df.to_csv(OUTPUT_DIR / 'feature_screen_lasso_rf.csv', index=False)
print('Saved:', OUTPUT_DIR / 'feature_screen_lasso_rf.csv')

## 7) Required Models

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1500,
            class_weight=class_weight_dict,
            random_state=RANDOM_STATE,
        )),
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=220,
        max_depth=8,
        min_samples_leaf=5,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=220,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='binary:logistic',
        eval_metric='auc',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    'Neural Network (MLP)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(
            hidden_layer_sizes=(48, 24),
            alpha=1e-4,
            max_iter=220,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

print('Models loaded:')
for k in models:
    print('-', k)

print('\nImbalance strategies used:')
print('- Logistic / RF: class_weight computed from data')
print('- XGBoost: scale_pos_weight = neg/pos')
print('- MLP: SMOTE on train set')


## 8) Quick Cross-Validation (Train Set)

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_models = {
    'Logistic Regression': clone(models['Logistic Regression']),
    'Random Forest': clone(models['Random Forest']),
    'XGBoost': clone(models['XGBoost']),
    'Neural Network (MLP)': ImbPipeline([
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('clf', clone(models['Neural Network (MLP)'])),
    ]),
}

cv_rows = []
for model_name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={'auc': 'roc_auc', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall'},
        n_jobs=-1,
        error_score='raise',
    )

    cv_rows.append({
        'model': model_name,
        'cv_auc_mean': round(float(np.mean(scores['test_auc'])), 4),
        'cv_f1_mean': round(float(np.mean(scores['test_f1'])), 4),
        'cv_precision_mean': round(float(np.mean(scores['test_precision'])), 4),
        'cv_recall_mean': round(float(np.mean(scores['test_recall'])), 4),
    })

cv_df = pd.DataFrame(cv_rows).sort_values('cv_auc_mean', ascending=False)
display(cv_df)

cv_df.to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
print('Saved:', OUTPUT_DIR / 'cv_results.csv')

## 9) Final Training + Test Evaluation

In [ ]:
results = []
report_rows = []
model_test_probs = {}
fitted_models = {}

# ROC figure
plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], linestyle='--', label='Random classifier')

# Confusion matrices figure (2x2)
fig_cm, axes_cm = plt.subplots(2, 2, figsize=(10, 8))
axes_cm = axes_cm.flatten()

for idx, (model_name, model) in enumerate(models.items()):
    if model_name == 'Neural Network (MLP)':
        model.fit(X_train_smote, y_train_smote)
    else:
        model.fit(X_train, y_train)

    fitted_models[model_name] = model

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= DEFAULT_THRESHOLD).astype(int)
    model_test_probs[model_name] = y_prob

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        'model': model_name,
        'auc': round(float(auc), 4),
        'f1_score': round(float(f1), 4),
        'precision': round(float(prec), 4),
        'recall': round(float(rec), 4),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    })

    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    for label, vals in rep.items():
        if isinstance(vals, dict):
            report_rows.append({
                'model': model_name,
                'label': label,
                'precision': vals.get('precision', np.nan),
                'recall': vals.get('recall', np.nan),
                'f1_score': vals.get('f1-score', np.nan),
                'support': vals.get('support', np.nan),
            })

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(ax=axes_cm[idx], colorbar=False)
    axes_cm[idx].set_title(f'{model_name} (thr={DEFAULT_THRESHOLD})')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Test set')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png')
plt.show()

fig_cm.tight_layout()
fig_cm.savefig(OUTPUT_DIR / 'confusion_matrices_grid.png')
plt.show()

results_df = pd.DataFrame(results).sort_values('auc', ascending=False)
report_df = pd.DataFrame(report_rows)
final_df = results_df.merge(cv_df, on='model', how='left')

display(final_df)

final_df.to_csv(OUTPUT_DIR / 'model_results.csv', index=False)
report_df.to_csv(OUTPUT_DIR / 'classification_report_full.csv', index=False)
print('Saved:', OUTPUT_DIR / 'model_results.csv')
print('Saved:', OUTPUT_DIR / 'classification_report_full.csv')

## 10) Threshold Trade-off (Best Model)

In [ ]:
best_model = final_df.sort_values('auc', ascending=False).iloc[0]['model']
best_probs = model_test_probs[best_model]

threshold_rows = []
for t in np.arange(0.25, 0.76, 0.05):
    y_pred_t = (best_probs >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)

    threshold_rows.append({
        'threshold': round(float(t), 2),
        'auc': round(float(roc_auc_score(y_test, best_probs)), 4),
        'f1': round(float(f1_score(y_test, y_pred_t)), 4),
        'precision': round(float(precision_score(y_test, y_pred_t, zero_division=0)), 4),
        'recall': round(float(recall_score(y_test, y_pred_t, zero_division=0)), 4),
        'tn': int(cm_t[0, 0]),
        'fp': int(cm_t[0, 1]),
        'fn': int(cm_t[1, 0]),
        'tp': int(cm_t[1, 1]),
    })

threshold_df = pd.DataFrame(threshold_rows)

candidates = threshold_df[threshold_df['precision'] >= 0.35]
if len(candidates) > 0:
    recommended = candidates.sort_values(['f1', 'recall'], ascending=False).iloc[0]
else:
    recommended = threshold_df.sort_values(['f1', 'recall'], ascending=False).iloc[0]
recommended_threshold = float(recommended['threshold'])

print(f'Best model: {best_model}')
print(f'Recommended threshold: {recommended_threshold:.2f}')
display(threshold_df)

plt.figure(figsize=(8, 5))
plt.plot(threshold_df['threshold'], threshold_df['f1'], label='F1')
plt.plot(threshold_df['threshold'], threshold_df['precision'], label='Precision')
plt.plot(threshold_df['threshold'], threshold_df['recall'], label='Recall')
plt.axvline(recommended_threshold, color='red', linestyle='--', label=f'Recommended={recommended_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title(f'Threshold trade-off - {best_model}')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'threshold_tradeoff_main.png')
plt.show()

threshold_df.to_csv(OUTPUT_DIR / 'threshold_analysis_best_model.csv', index=False)
print('Saved:', OUTPUT_DIR / 'threshold_analysis_best_model.csv')

## 11) Business Notes (Oral-Ready)

In [ ]:
best_row = final_df.sort_values('auc', ascending=False).iloc[0]
default_rate = float(df[target_col].mean())

notes = [
    f"Default rate is {default_rate:.1%}: this is an imbalanced classification problem.",
    "Defaulting clients show a weaker repayment profile: more delays and lower payment intensity vs bills.",
    f"Best model by AUC is {best_row['model']} (AUC={best_row['auc']:.3f}, F1={best_row['f1_score']:.3f}).",
    f"Recommended threshold is {recommended_threshold:.2f}: lower threshold improves recall but increases false positives.",
    "Small manual tuning was chosen to keep models interpretable and avoid overfitting in a student project.",
]

print('Business-ready summary:')
for i, n in enumerate(notes, 1):
    print(f'{i}. {n}')

feature_defense = {
    'util_ratio_mean': 'High utilization indicates financial stress.',
    'pay_to_bill_total_ratio': 'Low payment relative to bill suggests repayment difficulty.',
    'delinquency_count': 'Past delays are a direct signal of future risk.',
    'bill_trend_1_vs_6': 'A worsening bill trend may indicate rising leverage.',
}

print('\nFeature engineering defense (Q&A):')
for k, v in feature_defense.items():
    print(f'- {k}: {v}')

pd.DataFrame({'business_observation': notes}).to_csv(
    OUTPUT_DIR / 'business_observations_for_slides.csv',
    index=False,
)
pd.DataFrame(
    [{'feature': k, 'why_it_matters': v} for k, v in feature_defense.items()]
).to_csv(
    OUTPUT_DIR / 'feature_engineering_defense_notes.csv',
    index=False,
)
print('Saved:', OUTPUT_DIR / 'business_observations_for_slides.csv')
print('Saved:', OUTPUT_DIR / 'feature_engineering_defense_notes.csv')


## Output Files for Slides

- `outputs/eda_default_vs_non_default.csv`
- `outputs/feature_screen_lasso_rf.csv`
- `outputs/model_results.csv`
- `outputs/cv_results.csv`
- `outputs/classification_report_full.csv`
- `outputs/roc_curves.png`
- `outputs/confusion_matrices_grid.png`
- `outputs/threshold_analysis_best_model.csv`
- `outputs/threshold_tradeoff_main.png`
- `outputs/business_observations_for_slides.csv`
- `outputs/feature_engineering_defense_notes.csv`
